# Proyecto RL DEFI IV

**Integrantes**: Juan Camilo y Jesús Figueroa

**Descripción**:

Creación de un agente de aprendizaje por refuerzo para operar el token POL, empleando el histórico de transacciones on-chain y métricas financieras para identificar wallets con desempeño consistente, construir un perfil estadístico de su comportamiento e incorporar dicho perfil al proceso de toma de decisiones del agente, con el propósito de mejorar las decisiones de compra, venta o hold del activo.



El sistema reconstruirá el historial de actividad de las wallets seleccionadas, se usaran el precio de cierre de POL-USD en ventanas de 1m, 5m, 15m y 1h, con el objetivo de calcular la ganancia generada en esas ventanas.

## Guía de ejecución reproducible (paso a paso)

Esta guía permite ejecutar el proyecto desde Jupyter sin repetir descargas innecesarias. La ruta recomendada para la sustentación es reutilizar los archivos Parquet de `cache/`; sólo usa Alchemy cuando realmente necesites una ventana nueva de swaps.

### 1. Preparar el entorno

En una terminal abierta en la carpeta del repositorio ejecuta:

```powershell
py -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
jupyter lab
```

Abre este archivo (`Defi_IV.ipynb`) y selecciona el kernel de `.venv`. Después de instalar dependencias por primera vez, reinicia el kernel.

### 2. Elegir la fuente de datos

**Opción recomendada — sin red:** comprueba que existan `cache/swaps_24h.parquet` y, si vas a mostrar el resumen por wallet, `cache/wallets_24h.parquet`. Con `forzar_descarga=False`, el `SwapPipeline` reutiliza el cache y no consulta Alchemy.

**Opción para datos nuevos:** antes de ejecutar la configuración, define `ALCHEMY_RPC_URL` o `ALCHEMY_API_KEY` como variable de entorno. Si no hay Parquet, la celda de configuración solicitará la clave de forma oculta. Nunca pegues ni guardes la clave en el notebook ni en GitHub.

### 3. Ejecutar los perfiles de wallets

Ejecuta, en este orden: **Imports** → **Main Transacciones** (configuración y swaps) → **Main Wallets** → la celda que ejecuta `analizar_wallets(swaps)`. Esta procesa en memoria los swaps lógicos, ciclos FIFO y ledger de decisiones maduras, exponiendo `resultado.perfiles` y `resultado.ganadoras` sin sobrecarga en disco.

Luego ejecuta las celdas Buy, Sell y Hold para explorar los perfiles ganadores por horizonte. Una wallet entra como ganadora sólo si tiene decisiones maduras suficientes, PnL y retorno mediano positivos y el `consistency_score` requerido.

### 4. Ejecutar el agente Bellman

Desde **Parte III**, ejecuta las celdas en orden. La celda `ejecutar_agente_bellman` procesa directamente los DataFrames en memoria (`swaps_logicos`, `ledger`, `perfiles`) y resuelve el MDP mediante inducción hacia atrás sin consultar Alchemy ni depender de snapshots en disco.

### 5. Comprobar los resultados

El flujo opera 100% en memoria para máxima velocidad y reproducibilidad. Puedes inspeccionar las observaciones horarias, la matriz de transiciones verificada, la política óptima calculada y el replay interactivo directamente en el notebook.


## Imports

In [1]:
from lib_rl.transacciones import SwapPipeline
from lib_rl.wallets import WalletView
import warnings
warnings.filterwarnings('ignore')


## Main Transacciones

In [2]:
import os
from getpass import getpass
from pathlib import Path

POOL_ADDRESS = "0xA374094527e1673A86dE625aa59517c5dE346d32"  # WPOL/USDC 0.05%
TICKER_YF    = "POL28321-USD"
DECIMALES_T0 = 18   # WPOL
DECIMALES_T1 = 6    # USDC
NOMBRE_T0    = "pol"
NOMBRE_T1    = "usdc"

HORAS        = 24
HORIZONTES   = ["1m", "5m", "15m", "1h"]
CACHE_PATH   = "cache/swaps_24h.parquet"

# Con cache existente no se requiere conexión a Alchemy.
RPC_URL = os.environ.get("ALCHEMY_RPC_URL", "").strip()
if not RPC_URL:
    api_key = os.environ.get("ALCHEMY_API_KEY", "").strip()
    if not api_key and not Path(CACHE_PATH).is_file():
        api_key = getpass("Clave de Alchemy (no se guarda): ")
    if api_key:
        RPC_URL = f"https://polygon-mainnet.g.alchemy.com/v2/{api_key}"

if not Path(CACHE_PATH).is_file() and not RPC_URL:
    raise RuntimeError("No hay cache y falta ALCHEMY_RPC_URL o ALCHEMY_API_KEY.")


In [3]:
pipeline = SwapPipeline(
    rpc_url      = RPC_URL,
    pool_address = POOL_ADDRESS,
    ticker_yf    = TICKER_YF,
    decimales_t0 = DECIMALES_T0,
    decimales_t1 = DECIMALES_T1,
    nombre_t0    = NOMBRE_T0,
    nombre_t1    = NOMBRE_T1,
    cache_path   = CACHE_PATH,
)

swaps = pipeline.ejecutar(
    horas            = HORAS,
    horizontes       = HORIZONTES,
    forzar_descarga  = False,
    verbose          = True,
)


[cache] cargado <- cache\swaps_24h.parquet (21331 swaps)
[yfinance] descargando velas 1m...
[yfinance] descargando velas 5m...



1 Failed download:
['POL28321-USD']: YFPricesMissingError('possibly delisted; no price data found  (5m 2026-07-29 00:00:00+00:00 -> 2026-09-28 00:00:00+00:00) (Yahoo error = "5m data not available for startTime=1785283200 and endTime=1790549535. The requested range must be within the last 60 days.")')


  [warn] sin datos para 5m
[yfinance] descargando velas 15m...



1 Failed download:
['POL28321-USD']: YFPricesMissingError('possibly delisted; no price data found  (15m 2026-07-29 00:00:00+00:00 -> 2026-09-28 00:00:00+00:00) (Yahoo error = "15m data not available for startTime=1785283200 and endTime=1790549536. The requested range must be within the last 60 days.")')


  [warn] sin datos para 15m
[yfinance] descargando velas 1h...


In [4]:
# Tabla de Swaps
swaps.head()

,timestamp,bloque,hash_tx,wallet,direccion,pol_cantidad,usdc_cantidad,precio_ejecutado,gas_pol,gas_usdc,...,ganancia_usdc_1m,precio_fwd_5m,retorno_5m,ganancia_usdc_5m,precio_fwd_15m,retorno_15m,ganancia_usdc_15m,precio_fwd_1h,retorno_1h,ganancia_usdc_1h
0,2026-08-21 20:47:52-05:00,92438830,0xba45eff7940a3e13a3efe6650f54d9f9fd7995d84a24...,0x49fB304bA92f732486d23DE10d5824ff67A85F75,Sell,56.976172,5.453435,0.095714,0.240044,0.022976,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.09907,-0.035059,-0.191194
1,2026-08-21 20:48:36-05:00,92438859,0x8cc4fd5c05fd576cfd6d216e2231d7b89cfa8f915de9...,0xe27D4Cb1759BAE0188126bD62D415a6Fa528FFE5,Buy,210.691960,20.189782,0.095826,1.328537,0.127308,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.09907,0.033852,0.683470
2,2026-08-21 20:49:03-05:00,92438877,0x42520a2fd84fbe4ed8b9cc301cbb319a0497410ff00e...,0xeba3A54cffcC11123578C5A740a45456d5C6D4a8,Sell,222.132001,21.264494,0.095729,0.071096,0.006806,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.09907,-0.034900,-0.742123
3,2026-08-21 20:49:12-05:00,92438883,0x02a30b4fb5bcc347379ffecbe0211c4ec566e2d9db77...,0xeba3A54cffcC11123578C5A740a45456d5C6D4a8,Sell,191.333283,18.307962,0.095686,0.070256,0.006722,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.09907,-0.035363,-0.647426
4,2026-08-21 20:49:52-05:00,92438910,0x0087aed6f5d9d6a2d612db08ac0246dd39bb1eeff959...,0xeba3A54cffcC11123578C5A740a45456d5C6D4a8,Sell,224.353321,21.457538,0.095642,0.086444,0.008268,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.09907,-0.035845,-0.769145


## Main Wallets

In [5]:
# Para Historial de Wallets
wallets = WalletView(pipeline, horizontes=HORIZONTES)

In [6]:
# Historial y tabla agregada de wallets
from pathlib import Path

wallets.construir()
Path("cache").mkdir(exist_ok=True)
wallets.df.to_parquet("cache/wallets_24h.parquet", index=False)
print("Tabla de wallets guardada en cache/wallets_24h.parquet")


Tabla de wallets guardada en cache/wallets_24h.parquet


In [7]:
wallets.df

,wallet,direccion,n_swaps,pol_cantidad,usdc_cantidad,gas_usdc,usdc_neto,ganancia_usdc_1m,ganancia_usdc_5m,ganancia_usdc_15m,ganancia_usdc_1h
0,0x003AB61c1f180Eee803b9EA334B6b1B29A4c8364,Sell,1,777.660000,84.096490,0.029387,84.067103,0.0,0.0,0.0,0.124765
1,0x004125dc64f050C3CCEaD24A61c89aDDD4C2c794,Buy,1,111.208914,11.962802,0.047072,11.915730,0.0,0.0,0.0,-0.305884
2,0x0051391747B6C11110dA7dE76B3F4AED532d7E40,Buy,1,121.204415,12.272375,0.062058,12.210317,0.0,0.0,0.0,1.323125
3,0x007d9668D19418b66CD028cde1Ea598260152F93,Buy,1,13.590998,1.495507,0.042197,1.453310,0.0,0.0,0.0,-0.005390
4,0x0082CaF47363bD42917947d81f4d4E0395257267,Buy,1,137.456061,14.998583,0.181089,14.817494,0.0,0.0,0.0,0.419864
...,...,...,...,...,...,...,...,...,...,...,...
3053,0xfd50D4D48896e076A02Bd5F7534df1244948eAAE,Sell,1,282.750000,30.054951,0.012830,30.042121,0.0,0.0,0.0,-0.414188
3054,0xfeA573eF9d5eD5b470311582C99c103C38C0320f,Buy,1,0.002269,0.000243,0.020015,0.000000,0.0,0.0,0.0,-0.000007
3055,0xff7D510CF1ec1d376DB53e45bC754bDd8c041D3e,Buy,1,8.954345,1.032616,0.113928,0.918688,0.0,0.0,0.0,-0.019253
3056,0xff91d71CdbA995Bb2F2Ad009F2309E542036Ebd2,Buy,1,15.822873,1.739953,0.043565,1.696388,0.0,0.0,0.0,0.016069


In [ ]:
from lib_rl.wallets import analizar_wallets

# Análisis de perfiles y clasificación de wallets directamente en memoria
resultado = analizar_wallets(swaps, min_decisiones=3)
print(f"Perfiles analizados: {len(resultado.perfiles)} | Ganadoras: {len(resultado.ganadoras)}")
resultado.estados


## WALLETS GANADORAS 

In [ ]:
perfiles = resultado.perfiles
perfiles["winner_status"].value_counts()


In [ ]:
display(
    perfiles[perfiles["winner_status"] == "winner"]
    .sort_values("consistency_score", ascending=False)
)


## ACCION COMPRA 

In [ ]:
from lib_rl.wallets import filtrar_wallets_ganadoras

solo_buy_1m= filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["BUY_POL"],
    horizontes=["1m"],
)

display(solo_buy_1m)


In [ ]:
from lib_rl.wallets import filtrar_wallets_ganadoras

solo_buy_5m = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["BUY_POL"],
    horizontes=["5m"],
)

display(solo_buy_5m)


In [ ]:
from lib_rl.wallets import filtrar_wallets_ganadoras

solo_buy_15m = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["BUY_POL"],
    horizontes=["15m"],
)

display(solo_buy_15m)


In [ ]:
from lib_rl.wallets import filtrar_wallets_ganadoras

solo_buy_1h = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["BUY_POL"],
    horizontes=["1h"],
)

display(solo_buy_1h)


## ACCION POR VENTA 

In [ ]:
from lib_rl.wallets import filtrar_wallets_ganadoras

solo_sell_1m = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["SELL_POL"],
    horizontes=["1m"],
)

display(solo_sell_1m)


In [ ]:
from lib_rl.wallets import filtrar_wallets_ganadoras

solo_sell_5m = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["SELL_POL"],
    horizontes=["5m"],
)

display(solo_sell_5m)


In [ ]:
from lib_rl.wallets import filtrar_wallets_ganadoras

solo_sell_15m = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["SELL_POL"],
    horizontes=["15m"],
)

display(solo_sell_15m)


In [ ]:
from lib_rl.wallets import filtrar_wallets_ganadoras

solo_sell_1h = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["SELL_POL"],
    horizontes=["1h"],
)

display(solo_sell_1h)


## ACCION HOLD 

In [ ]:
from lib_rl.wallets import filtrar_wallets_ganadoras

solo_hold_1m = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["HOLD"],
    horizontes=["1m"],
)

display(solo_hold_1m)


In [ ]:
from lib_rl.wallets import filtrar_wallets_ganadoras

solo_hold_5m = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["HOLD"],
    horizontes=["5m"],
)

display(solo_hold_5m)


In [ ]:
from lib_rl.wallets import filtrar_wallets_ganadoras

solo_hold_15m = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["HOLD"],
    horizontes=["15m"],
)

display(solo_hold_15m)


In [ ]:
from lib_rl.wallets import filtrar_wallets_ganadoras

solo_hold_1h = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["HOLD"],
    horizontes=["1h"],
)

display(solo_hold_1h)


In [ ]:
# Perfiles clasificados como ganadores
solo_winners = resultado.ganadoras
print(f"Total perfiles ganadores: {len(solo_winners)}")


In [ ]:
solo_winners= resultado.perfiles[resultado.perfiles["winner_status"]=="winner"]
solo_winners


# Parte III. Modelo de aprendizaje por refuerzo guiado por wallets

Esta sección transforma los perfiles on-chain ya calculados en un modelo de decisión. El objetivo no es copiar una wallet, sino aprender una política que use su evidencia histórica como una señal adicional para operar POL cada hora.

## 1. Definición teórica: proceso de decisión de Markov

Un problema de aprendizaje por refuerzo se representa como un **Proceso de Decisión de Markov (MDP)**:

$$M=(S,A,P,R,\gamma)$$

- $S$: conjunto de estados observables del mercado y del portafolio.
- $A$: acciones que puede tomar el agente.
- $P(s'|s,a)$: probabilidad de pasar al estado siguiente $s'$ después de elegir $a$ en $s$.
- $R(s,a,s')$: recompensa obtenida por la decisión.
- $\gamma$: factor de descuento; valora las recompensas futuras sin ignorar la recompensa inmediata.

Aquí el **agente** es una estrategia automática de POL; el **entorno** es el mercado POL/USDC observado on-chain; una decisión ocurre cada **1 hora**; y el episodio tiene inicialmente 24 decisiones.

## 2. Wallets ganadoras como señal, no como regla fija

Primero se construyen perfiles por `wallet × acción × horizonte`. Para el agente sólo se usan perfiles de **1 h** que ya son ganadores y tienen un umbral alto:

$$winner\_status=winner \quad \text{y} \quad consistency\_score \ge 0.80$$

El score resume tres evidencias: tasa de acierto, calidad de ganancias frente a pérdidas y número de decisiones maduras:

$$consistency = 0.50\cdot tasa\_acierto + 0.30\cdot\frac{min(profit\_factor,3)}{3} + 0.20\cdot min(n/10,1)$$

Una decisión está madura si ya llegó su precio posterior de una hora. En el corte $t$, el sistema sólo usa decisiones con precio forward conocido antes o en $t$; por eso la señal no usa información futura. Se suman los scores de las wallets Buy, Sell y Hold. La dirección con soporte único más alto genera `BUY`, `SELL` o `HOLD`; un empate o ausencia de evidencia genera `NEUTRAL`.

## 3. Estado y acciones del agente

El estado combina tres elementos:

$$s_t=(r_t,w_t,x_t)$$

- $r_t$: régimen de mercado `DOWN`, `FLAT` o `UP`. Se construye a partir de retornos pasados de 1m, 5m, 15m y 1h. Una banda fija de ±0.10 % define `FLAT`, para no depender de precios futuros.
- $w_t$: señal causal de wallets: `BUY`, `SELL`, `HOLD` o `NEUTRAL`.
- $x_t$: posición actual: `0 = USDC` y `1 = POL`.

Por tanto, hay $3\times4\times2=24$ estados posibles. El modelo es **long-only**: no vende POL que no posee ni abre posiciones cortas.

| Posición actual | Acciones admisibles | Posición siguiente |
|---|---|---|
| USDC (`0`) | `BUY_POL`, `HOLD` | POL o USDC |
| POL (`1`) | `SELL_POL`, `HOLD` | USDC o POL |

## 4. Transiciones empíricas

No se inventan probabilidades. Se observan los cambios horarios consecutivos de $(r_t,w_t)$ en los datos y se cuentan. Después se aplica suavizado de Laplace para que una transición no observada no tenga probabilidad exactamente cero:

$$P(s'|s,a)=\frac{conteo(s\rightarrow s')+\alpha}{\sum_j conteo(s\rightarrow s_j)+\alpha\cdot|S_{económico}|}$$

con $\alpha=1$. La acción determina la posición siguiente: por ejemplo, desde USDC, `BUY_POL` lleva a posición POL; desde POL, `SELL_POL` lleva a USDC. En cada estado y acción admisible verificamos que $\sum_{s'}P(s'|s,a)=1$.

## 5. Función de recompensa

La recompensa principal es siempre financiera. Si $p_t$ es el precio actual y $p_{t+1}$ el precio conocido una hora después, el retorno de POL es:

$$retorno_{POL}=\frac{p_{t+1}}{p_t}-1$$

La recompensa base es el retorno de la posición que queda después de la acción, menos gas al comprar o vender. Así, desde USDC: `BUY_POL` obtiene $retorno_{POL}-gas$ y `HOLD` obtiene 0. Desde POL: `SELL_POL` obtiene $-gas$ y `HOLD` obtiene $retorno_{POL}$. El precio $p_{t+1}$ sólo se utiliza sobre el historial para calcular recompensas y estimar el modelo; en una decisión real el agente sólo observa la información disponible en $t$.

Las wallets no entregan dinero ficticio sólo por seguirlas. Añaden un bonus pequeño únicamente si su dirección dominante coincide con una decisión que efectivamente fue mejor que su alternativa:

$$R_{final}=R_{base}+0.25\cdot confianza_{wallets}\cdot ventaja_{realizada}$$

La confianza es mayor cuando el soporte de una dirección supera claramente a las otras. La ventaja compara la recompensa base de una acción contra la otra acción admisible.

## 6. Política óptima mediante Bellman

Como el episodio tiene horizonte finito de las horas completas disponibles, se resuelve Bellman hacia atrás, desde la última hora hasta la primera:

$$V_t(s)=\max_{a\in A(s)}\left[R(s,a)+\gamma\sum_{s'}P(s'|s,a)V_{t+1}(s')\right]$$

La acción que alcanza el máximo define la política $\pi_t^*(s)$. En palabras sencillas: para cada estado, el agente compara las acciones permitidas, combina la ganancia inmediata con el valor esperado de las siguientes horas y elige la mejor. Usamos $\gamma=0.99$, por lo que el futuro importa casi tanto como la hora actual.

## 7. Ejemplo numérico sencillo

Supón que el agente está en USDC, POL vale $1.00$ ahora y $1.02$ una hora después. El retorno de POL es 2 %. Comprar cuesta 0.10 % de gas. Las wallets confirmadas suman soporte Buy = 0.90, Sell = 0.10 y Hold = 0.00.

La confianza de Buy es $(0.90-0.10)/(0.90+0.10)=0.80$. Las dos acciones permitidas son:

| Acción | Recompensa base | Ventaja frente a la alternativa | Bonus wallets | Recompensa final |
|---|---:|---:|---:|---:|
| `BUY_POL` | $0.02-0.001=0.019$ | $0.019-0=0.019$ | $0.25\times0.80\times0.019=0.0038$ | $0.0228$ |
| `HOLD` en USDC | $0$ | $0-0.019=-0.019$ | $0$ | $0$ |

En esta hora Buy es mejor porque el precio subió y las wallets Buy tenían apoyo dominante. Si el precio hubiera caído, la recompensa base de comprar sería negativa y el bonus tampoco convertiría esa mala decisión en una buena.

## 8. Interpretación y límites

Las siguientes celdas ejecutan el modelo con los datos disponibles y muestran la cohorte de wallets, los estados, la verificación de transiciones, la política y un replay histórico. El replay es una explicación **in-sample**, no una prueba de rentabilidad real. La ventana de 24 horas es aún pequeña para estimar una política estable; el siguiente paso es acumular snapshots, separar entrenamiento/validación/prueba en orden temporal y evaluar fuera de muestra. El modelo tampoco representa balances externos, posiciones cortas ni impacto de mercado.

In [ ]:
winner_filtrados = solo_winners[solo_winners["consistency_score"]>=0.8]
winner_filtrados["horizonte"].value_counts()

In [ ]:
winner_filtrados[["horizonte","retorno_neto_mediano"]].groupby(by="horizonte").mean()


In [ ]:
# 1. Wallets ganadoras dirigidas: sólo 1h y consistency_score >= 0.80
from lib_rl.wallets import calcular_wallets_ganadoras_1h_consistentes

UMBRAL_CONSISTENCIA = 0.80

wallets_dirigidas_1h = calcular_wallets_ganadoras_1h_consistentes(
    resultado.perfiles,
    consistency_threshold=UMBRAL_CONSISTENCIA,
)

display(wallets_dirigidas_1h)
print("Wallets únicas:", wallets_dirigidas_1h["wallet"].nunique())


In [ ]:
# 2. Construir el MDP y resolver Bellman directamente en memoria
from lib_rl.bellman import ejecutar_agente_bellman

resultado_rl = ejecutar_agente_bellman(
    swaps_logicos=resultado.swaps_logicos,
    ledger=resultado.ledger,
    perfiles=resultado.perfiles,
    consistency_threshold=0.80,
    flat_band=0.001,
    profile_weight=0.25,
    horizon=None,  # tantas etapas como horas completas observadas
    gamma=0.99,
)

In [ ]:
# 3. Estados horarios: mercado + señal de wallets + posición
observaciones_rl = resultado_rl.observaciones

display(
    observaciones_rl[
        [
            "as_of", "precio_t", "precio_t1",
            "regimen_mercado", "senal_wallets",
            "confianza_wallets"
        ]
    ]
)

In [ ]:
# 4. Verificar que P(s' | s, a) suma 1
from lib_rl.bellman import verificar_probabilidades

verificacion = verificar_probabilidades(resultado_rl.mdp)
display(verificacion)

assert verificacion["valida"].all()
print("Todas las transiciones válidas suman 1.")


In [ ]:
# 5. Política óptima calculada con Bellman
politica = resultado_rl.politica

display(
    politica.sort_values(
        ["posicion", "regimen_mercado", "senal_wallets"]
    )
)

In [ ]:
# 6. Replay histórico: qué habría decidido el agente cada hora
replay = resultado_rl.replay

display(replay)
print("Riqueza final, base 100:", replay["wealth"].iloc[-1])

In [ ]:
import plotly.express as px

fig_senal = px.scatter(
    observaciones_rl,
    x="as_of",
    y="confianza_wallets",
    color="senal_wallets",
    hover_data=["support_buy", "support_sell", "support_hold"],
    title="Señal de wallets ganadoras por hora",
)

fig_replay = px.line(
    replay,
    x="as_of",
    y="wealth",
    markers=True,
    title="Replay histórico de la política Bellman",
)

fig_senal.show()
fig_replay.show()


# Parte IV. Q-learning con la misma evidencia de wallets

Q-learning conserva el activo POL, la frecuencia de una hora, el estado `(régimen, señal de wallets, posición)`, las acciones admisibles y la recompensa neta de gas del modelo Bellman. La diferencia es que aprende valores Q desde experiencias históricas, sin usar la matriz de transición durante la actualización.

Para cada experiencia `(S_t, A_t, R_{t+1}, S_{t+1})`, la actualización combina la recompensa observada con el mejor valor disponible en el estado siguiente. Durante el entrenamiento se usa exploración epsilon-greedy; en el replay Q queda congelada.

Esta parte sigue la misma estructura paso a paso de la Parte III: primero la teoría de Q-learning y en qué se diferencia de Bellman, luego cómo se genera el entrenamiento en este proyecto concreto, y finalmente la política óptima encontrada junto con la secuencia de acciones que esa política habría tomado hora a hora.

## 1. Aprendizaje por refuerzo *model-free*: la diferencia con Bellman

Bellman (Parte III) es un método **model-based**: primero se estima explícitamente la matriz de transición empírica $P(s'\mid s,a)$ y la recompensa esperada $R(s,a)$ para los 24 estados, y después se resuelve exactamente la ecuación de Bellman por inducción hacia atrás.

Q-learning es **model-free**: nunca calcula ni usa $P(s'\mid s,a)$. En su lugar aprende una tabla de valores $Q(s,a)$ directamente de experiencias observadas $(s_t, a_t, r_{t+1}, s_{t+1})$, actualizándola poco a poco mediante **diferencias temporales** (*temporal-difference learning*, TD).

Todo lo demás del modelo se conserva sin cambios respecto a la Parte III:

| Elemento | ¿Cambia en Q-learning? |
|---|---|
| Estado $s=(r_t,w_t,x_t)$ | No, es el mismo de la sección 3 (Parte III). |
| Acciones admisibles | No, las mismas reglas long-only de la sección 10 (Parte III). |
| Recompensa $R_{final}$ | No, la misma función con bonus de wallets de la sección 5 (Parte III). |
| $\gamma$ (descuento) | No, se usa 0.99, igual que Bellman. |
| Cómo se calcula la política óptima | **Sí.** Bellman resuelve una ecuación exacta; Q-learning la aproxima repitiendo experiencia. |

En pocas palabras: Q-learning responde la misma pregunta que Bellman (¿qué acción maximiza el valor esperado?), pero sin conocer de antemano las probabilidades de transición; las descubre implícitamente al repetir la experiencia muchas veces.

## 2. La ecuación de actualización de Q-learning

Cada vez que el agente vive una experiencia $(s_t, a_t, r_{t+1}, s_{t+1})$, actualiza un único valor de su tabla:

$$Q(s_t,a_t)\;\leftarrow\;Q(s_t,a_t)+\alpha\Big[\underbrace{r_{t+1}+\gamma\max_{a'}Q(s_{t+1},a')}_{\text{objetivo TD}}-\underbrace{Q(s_t,a_t)}_{\text{valor actual}}\Big]$$

- $\alpha\in(0,1]$ es la **tasa de aprendizaje**: qué tanto peso se le da a la nueva observación frente al valor ya almacenado. En este notebook $\alpha=0.15$.
- El **objetivo TD** combina la recompensa realmente observada $r_{t+1}$ con el mejor valor conocido en el estado siguiente, descontado por $\gamma$. Es el equivalente muestral de $R(s,a)+\gamma\sum_{s'}P(s'\mid s,a)V(s')$ en Bellman, pero usando el estado siguiente que **realmente ocurrió** en vez del promedio ponderado por todas las probabilidades.
- El **error TD** (la diferencia entre el objetivo y el valor actual) mide qué tan distinta es la experiencia nueva de lo que la tabla ya creía; si es cero, la tabla ya está en equilibrio para esa transición.

Como el horizonte del proyecto es finito (las horas completas disponibles, igual que Bellman), la tabla no es $Q(s,a)$ sino $Q_t(s,a)$: depende también de la hora exacta $t$ dentro del horizonte, igual que $V_t$ en la sección 6 (Parte III). Por eso `resultado_q.q_values` tiene forma `(horas, estados, acciones)` y no sólo `(estados, acciones)`.

## 3. Cómo se genera la experiencia en este proyecto: repetir la misma historia muchas veces

Sólo existe **una** ventana histórica real, cercana a 24 horas (la misma que usa Bellman). Como Q-learning necesita muchas experiencias para converger, este notebook genera esas experiencias repitiendo esa misma ventana 2.000 veces; cada repetición se llama **episodio**.

Al iniciar cada episodio, el agente vuelve a `posición = 0` (todo en USDC) y recorre las horas completas **en el mismo orden y con los mismos precios y señales de wallets que realmente ocurrieron**. Lo único que puede variar de un episodio a otro es qué acción explora el agente en cada hora y, por lo tanto, qué posición (USDC o POL) arrastra hacia la hora siguiente dentro de ese episodio.

Esto tiene una consecuencia importante para leer los resultados más adelante: en la hora $t$, el régimen de mercado y la señal de wallets **siempre son los mismos** (vienen fijos de los datos históricos); lo único que puede cambiar es la posición. Por lo tanto, de los 24 estados abstractos posibles, en cada hora sólo se pueden visitar **como máximo 2** (uno por cada valor de posición) — y en la primera hora ($t=0$) sólo se visita **uno solo**, porque todo episodio arranca en USDC. Los estados que ninguna trayectoria explorada llega a visitar nunca se actualizan y quedan con su valor inicial $Q=0$; esto se vuelve visible más adelante, en la tabla de política.

## 4. Exploración epsilon-greedy con decaimiento

En cada hora, el agente elige su acción con la regla **epsilon-greedy**:

$$a_t=\begin{cases}\text{acción aleatoria entre las admisibles} & \text{con probabilidad }\varepsilon\\[4pt]\arg\max_a Q_t(s_t,a) & \text{con probabilidad }1-\varepsilon\end{cases}$$

$\varepsilon$ no es constante: decrece linealmente episodio a episodio, desde $\varepsilon_{inicio}$ hasta $\varepsilon_{final}$:

$$\varepsilon(e)=\varepsilon_{inicio}+\frac{e-1}{E-1}\big(\varepsilon_{final}-\varepsilon_{inicio}\big)$$

con $e$ el número de episodio y $E$ el total de episodios. Al principio del entrenamiento $\varepsilon$ es alto: el agente explora mucho, probando compras y ventas en horas donde nunca lo haría por su cuenta, lo que le permite conocer el valor de posiciones que de otro modo jamás visitaría (por ejemplo, quedarse en POL en una hora donde lo natural sería estar en USDC). Hacia el final, $\varepsilon$ es bajo: el agente casi siempre explota lo que ya sabe, lo que estabiliza la tabla $Q$ alrededor de su mejor estimación.

## 5. Hiperparámetros usados en este notebook

| Parámetro | Valor | Qué controla |
|---|---:|---|
| `episodes` | 2.000 | Cuántas veces se repite la ventana histórica de horas completas. |
| `alpha` | 0.15 | Tasa de aprendizaje: peso de cada nueva observación sobre el valor almacenado. |
| `gamma` | 0.99 | Descuento del valor futuro; el mismo que usa Bellman, para que ambas políticas sean comparables. |
| `epsilon_start` | 0.90 | Probabilidad de explorar al inicio del entrenamiento. |
| `epsilon_end` | 0.05 | Probabilidad de explorar al final del entrenamiento. |
| `seed` | 7 | Semilla del generador aleatorio, para que el entrenamiento sea reproducible. |

Con $N$ horas completas y 2.000 episodios se realizan $N\times2.000$ actualizaciones de la tabla $Q$ (una por cada hora de cada episodio) — aunque, como se explicó arriba, se concentran en muy pocos pares (hora, estado).

## 6. Ejemplo numérico paso a paso de una actualización Q

Se reutiliza el mismo escenario del ejemplo de Bellman (sección 7, Parte III): el agente está en USDC, POL sube de 1.00 a 1.02, comprar cuesta 0.10 % de gas y la confianza de la señal Buy es 0.80. Ya se calculó allí que la recompensa final de comprar en esa hora es:

$$r_{t+1}=R_{final}(\text{BUY\_POL})=0.0228$$

Supongamos que la tabla $Q$ todavía no ha visitado este par (hora, estado, acción), así que $Q_t(s,\text{BUY\_POL})=0$, y que el mejor valor conocido en la hora siguiente (ya en posición POL) es $\max_{a'}Q_{t+1}(s',a')=0.03$. Con $\alpha=0.15$ y $\gamma=0.99$:

$$\text{objetivo TD}=0.0228+0.99\times0.03=0.0525$$
$$\text{error TD}=0.0525-0=0.0525$$
$$Q_{nuevo}=0+0.15\times0.0525=0.0079$$

Una sola actualización sólo mueve $Q$ un 15 % hacia el objetivo observado; no lo iguala de inmediato. Si esta misma hora se repite en otro episodio con un resultado similar, $Q$ se acerca un poco más cada vez, hasta estabilizarse alrededor del promedio de los objetivos observados — ese es exactamente el papel de repetir la historia miles de veces.

## 7. Entrenamiento: 2.000 episodios sobre la misma ventana histórica

La siguiente celda ejecuta exactamente el procedimiento descrito arriba: recorre las horas completas históricas 2.000 veces, con exploración epsilon-greedy decreciente, actualizando la tabla $Q_t(s,a)$ con la ecuación de la sección 2.

In [ ]:
# 1. Entrenar Q-learning con las mismas observaciones construidas para Bellman
from lib_rl.qlearning import entrenar_q_learning, tabla_politica_q_learning

Q_EPISODIOS = 2_000
resultado_q = entrenar_q_learning(
    observaciones_rl,
    resultado_rl.mdp,
    episodes=Q_EPISODIOS,
    alpha=0.15,
    gamma=0.99,
    epsilon_start=0.90,
    epsilon_end=0.05,
    seed=7,
)

display(resultado_q.training_log.tail())
print(f"Actualizaciones realizadas: {resultado_q.visits.sum():,}")
print("Forma de Q (hora, estado, acción):", resultado_q.q_values.shape)


### Cómo leer el registro de entrenamiento

`training_log` guarda, episodio a episodio, la recompensa total acumulada en esa repetición de las horas completas y el promedio de $|Q|$ en toda la tabla. La recompensa total **no crece de forma monótona**: como $\varepsilon$ nunca llega a cero, siempre queda algo de exploración aleatoria que puede empeorar un episodio puntual. Lo que sí debería estabilizarse hacia el final es `q_abs_media`: si deja de moverse, la tabla ya no está aprendiendo nada nuevo de las (pocas) transiciones que efectivamente visita.

In [ ]:
# 2. Política greedy aprendida y cobertura de acciones por estado
politica_q = tabla_politica_q_learning(resultado_rl.mdp, resultado_q, time=0)
display(politica_q.sort_values(["posicion", "regimen_mercado", "senal_wallets"]))

visitas_q = politica_q[[
    "regimen_mercado", "senal_wallets", "posicion",
    "visitas_buy_pol", "visitas_sell_pol", "visitas_hold",
]]
display(visitas_q)


## 8. Cómo leer la política aprendida: estados visitados y estados nunca observados

La tabla anterior muestra, para $t=0$, la acción con mayor $Q$ en cada uno de los 24 estados abstractos — pero no todos esos estados ocurrieron alguna vez en el entrenamiento. Como se explicó en la sección 3, en la primera hora sólo existe un estado real posible (el régimen y la señal de esa hora concreta, con posición = 0). Ese es el único renglón de la tabla con `visitas > 0` y un `valor_q` distinto de cero; todos los demás renglones muestran `valor_q = 0.0` y `visitas = 0` porque esa combinación de régimen, señal y posición nunca fue alcanzada a esa hora exacta durante el entrenamiento. Su "acción recomendada" no refleja entonces ninguna preferencia aprendida: es simplemente la primera acción admisible en caso de empate en cero.

Esto es una limitación real de este diseño, no un error de la implementación: al no usar ninguna matriz de transición para generalizar (a diferencia de Bellman), Q-learning sólo puede aprender sobre los pares (hora, estado) que efectivamente recorre la única trayectoria histórica disponible. La siguiente sección recupera exactamente esos pares y muestra la política aplicada hora por hora sobre la historia real, que es donde sí hay evidencia genuina.

## 9. Política óptima aplicada en el tiempo: la secuencia de acciones

En vez de mirar los 24 estados abstractos, aquí se aplica la política *greedy* aprendida (la tabla $Q$ ya congelada, sin más exploración) sobre la secuencia real de horas completas: en cada hora se observa el estado que realmente ocurrió y se elige $\arg\max_a Q_t(s,a)$. Esto responde directamente qué habría hecho el agente, hora a hora, siguiendo lo que aprendió — igual que el replay histórico de Bellman en la sección 6 (Parte III), pero con la política de Q-learning.

In [ ]:
# 3. Secuencia de acciones en el tiempo: replay de la política greedy aprendida
from lib_rl.qlearning import replay_q_learning_historico
import plotly.express as px

replay_q = replay_q_learning_historico(observaciones_rl, resultado_rl.mdp, resultado_q)

display(
    replay_q[[
        "as_of", "estado", "accion", "posicion_despues",
        "recompensa_final", "wealth",
    ]]
)
print("Política óptima de Q-learning aplicada sobre la historia real, hora a hora.")

fig_acciones_q = px.scatter(
    replay_q,
    x="as_of",
    y="posicion_despues",
    color="accion",
    title="Secuencia de acciones de Q-learning en el tiempo (0 = USDC, 1 = POL)",
)
fig_acciones_q.update_yaxes(tickvals=[0, 1], ticktext=["USDC (0)", "POL (1)"])
fig_acciones_q.show()

## 10. Comparación con Bellman y Buy and Hold

La comparación muestra dos medidas. `wealth` es la recompensa objetivo acumulada de los agentes e incluye el ajuste de wallets. `wealth_economica` usa sólo el retorno de la posición menos gas, por lo que es la medida comparable con Buy and Hold. El replay es explicativo e in-sample; para evaluar desempeño se debe entrenar y probar en bloques temporales separados.

In [ ]:
# 4. Comparación final: Bellman, Q-learning y Buy and Hold
from lib_rl.qlearning import replay_buy_and_hold, replay_q_learning_historico
import pandas as pd
import plotly.express as px

replay_q = replay_q_learning_historico(observaciones_rl, resultado_rl.mdp, resultado_q)
replay_buy_hold = replay_buy_and_hold(observaciones_rl)
replay_bellman = replay.copy()
replay_bellman["wealth_economica"] = 100.0 * (1.0 + replay_bellman["recompensa_base"]).cumprod()

comparacion_final = pd.DataFrame([
    {"estrategia": "Bellman", "riqueza_economica_final": replay_bellman["wealth_economica"].iloc[-1], "riqueza_objetivo_final": replay_bellman["wealth"].iloc[-1]},
    {"estrategia": "Q-learning", "riqueza_economica_final": replay_q["wealth_economica"].iloc[-1], "riqueza_objetivo_final": replay_q["wealth"].iloc[-1]},
    {"estrategia": "Buy and Hold", "riqueza_economica_final": replay_buy_hold["wealth_economica"].iloc[-1], "riqueza_objetivo_final": pd.NA},
])
display(comparacion_final.round(4))

trayectorias = pd.concat([
    replay_bellman[["as_of", "wealth_economica"]].assign(estrategia="Bellman"),
    replay_q[["as_of", "wealth_economica"]].assign(estrategia="Q-learning"),
    replay_buy_hold[["as_of", "wealth_economica"]].assign(estrategia="Buy and Hold"),
])
px.line(trayectorias, x="as_of", y="wealth_economica", color="estrategia", markers=True, title="Comparación in-sample: riqueza económica neta de gas").show()
px.line(resultado_q.training_log, x="episode", y="recompensa_total", title="Q-learning: recompensa por episodio durante el entrenamiento").show()


## 11. Limitaciones específicas de este Q-learning

Además de las limitaciones generales ya señaladas para Bellman (sección 8, Parte III: poca historia, evaluación in-sample, modelo long-only simplificado), esta implementación de Q-learning tiene limitaciones propias de su diseño:

1. **Una sola trayectoria repetida, no episodios independientes.** Los 2.000 "episodios" no son 2.000 muestras distintas de mercado: son la misma secuencia real de horas completas recorrida una y otra vez. Q-learning aquí no generaliza a mercados alternativos, sólo explora distintos caminos de posición sobre el único camino de precios que existe.
2. **La tabla $Q_t(s,a)$ está mayormente vacía.** Como se explicó en la sección 3 y en la sección 8, en cada hora sólo se visitan uno o dos de los 24 estados abstractos posibles. La política de la sección 9 (aplicada sobre la historia real) es la parte confiable del resultado; la tabla completa de la sección 8 mezcla valores aprendidos con ceros de inicialización nunca visitados.
3. **Sin matriz de transición, no hay forma de interpolar entre estados parecidos.** Bellman puede opinar sobre un estado nunca observado usando la transición empírica hacia estados vecinos; esta versión de Q-learning no tiene ese mecanismo, por lo que su cobertura del espacio de estados depende enteramente de cuántas combinaciones distintas aparezcan en los datos disponibles.

El siguiente paso natural, igual que para Bellman, es acumular varias ventanas históricas (distintos días u horas) para que cada episodio represente una experiencia de mercado realmente distinta, y así permitir que Q-learning aprenda una tabla $Q(s,a)$ que generalice más allá de una sola trayectoria.

# Parte V. Deep Q-Learning (DQN): teoría, aplicación a POL y evaluación

## 1. Del Bellman tabular a una red neuronal

Bellman estima una matriz de transición y resuelve el MDP. Q-learning actualiza una celda $Q_t(s,a)$ con cada experiencia. DQN mantiene el objetivo de Bellman, pero aproxima los valores con $Q_\theta(x,a)$: una red comparte parámetros entre estados. Sus salidas son valores estimados, no probabilidades.

$$y=r+\gamma(1-done)\max_{a'\in A(s')}Q_{\theta^-}(x',a')$$

El buffer guarda experiencias; un minilote las mezcla; Huber mide la diferencia entre $Q_\theta(x,a)$ y $y$; Adam ajusta la red principal y la copia objetivo se sincroniza cada 120 actualizaciones. El máximo futuro se restringe a acciones válidas. Al finalizar, `done=1` elimina el término futuro.


In [ ]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 2.4))
pasos = [("Estado", "9 entradas"), ("Capa 1", "32 + ReLU"),
         ("Capa 2", "32 + ReLU"), ("Valores Q", "3 acciones")]
for i, (titulo, detalle) in enumerate(pasos):
    ax.text(i, 0, f"{titulo}\n{detalle}", ha="center", va="center",
            fontsize=11, bbox=dict(boxstyle="round,pad=.8", facecolor="#dbeafe", edgecolor="#2563eb"))
    if i:
        ax.annotate("", xy=(i-.25,0), xytext=(i-.75,0),
                    arrowprops=dict(arrowstyle="->", color="#334155", lw=1.8))
ax.set(xlim=(-.6, 3.6), ylim=(-1,1), title="DQN: del estado a los valores Q")
ax.axis("off")
plt.tight_layout()
plt.show()


## 2. Nuestro problema: POL/USDC y evidencia de wallets

Se mantienen POL, decisiones horarias, posición binaria USDC/POL, acciones BUY_POL, SELL_POL y HOLD, costo de gas y recompensa con evidencia de wallets de las Partes III y IV. No usamos Short ni funding del ejercicio de futuros. Desde USDC son válidas BUY y HOLD; desde POL, SELL y HOLD. Se enmascaran las demás acciones al elegir y en el target.

La entrada de nueve números codifica régimen (3 categorías), señal de wallets (4), posición (1) y etapa normalizada (1). Esta representación conserva la información categórica de la tabla y permite compartir parámetros; no demuestra que una red supere la tabla. precio_t1 solo interviene después de actuar para calcular la recompensa.

La recompensa de aprendizaje mantiene reward_final = reward_base + profile_bonus; reward_base es el retorno económico de la posición menos gas. El bonus es una preferencia del objetivo, **no dinero adicional**. Por eso reportamos la riqueza económica por separado de la riqueza del objetivo. Como en las partes anteriores no se impone una venta final forzada; el valor final de la posición queda marcado a mercado.

## 3. Diseño experimental y alcance

El cache disponible contiene aproximadamente 24 horas, por lo que no permite tres conjuntos independientes de episodios de 24 horas. Para esta prueba exploratoria usamos bloques cronológicos de **4 horas**, con posición inicial en USDC: los primeros bloques completos son train, el penúltimo validación y el último test. Se estima el costo de gas únicamente con swaps anteriores a validación. La prueba descarta horas sobrantes y detiene la ejecución si hay menos de tres bloques completos o huecos horarios.

Comparamos dos valores de exploración final del DQN (0.05 y 0.30), con la misma semilla; elegimos por retorno económico de validación. Después evaluamos una sola vez en test las políticas congeladas DQN, Q tabular, Bellman y Buy and Hold. Tabla y DQN usan los mismos bloques de entrenamiento; Bellman estima su MDP allí. La prueba de estrés apaga las señales de wallets en el mismo test, sin reentrenar; es una intervención contrafactual, no otro conjunto independiente. La selección con un único bloque de validación es muy incierta.

**Control temporal:** para construir la señal, una experiencia HOLD solo se usa cuando su venta de cierre ya se conoce. Se usan precios de swaps observados hasta el instante de evaluación y se excluyen horas incompletas. La muestra sigue siendo corta y se requieren más ventanas independientes.


In [ ]:
# Requiere PyTorch: en Colab, si es necesario, ejecutar %pip install torch.
from lib_rl.dqn import experimento_pol, resumen_resultados, auditar

experimento_dqn = experimento_pol(observaciones_rl, resultado.swaps_logicos,
                                 horizonte=4, episodios=600, seed=7)
print('Bloques train/validación/test:', len(experimento_dqn['train']), 1, 1)
print('Gas estimado antes de validación:', experimento_dqn['gas'])
for nombre in ('train', 'validacion_bloques', 'test'):
    bloques = experimento_dqn[nombre]
    print(nombre, bloques[0].as_of.iloc[0], '→', bloques[-1].as_of.iloc[-1])
display(experimento_dqn['validacion'])
print('Exploración final elegida con validación:', experimento_dqn['epsilon_elegido'])


## 4. Evidencia de entrenamiento y transición

La recompensa del episodio incluye el bonus; la curva de Huber es un error de predicción, no rentabilidad. El buffer y las redes se mantienen congelados en evaluación. La auditoría usa una transición real de entrenamiento y los parámetros finales disponibles; no reconstruye los parámetros históricos del instante en que se guardó.


In [ ]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

modelo_dqn = experimento_dqn['modelos']['DQN']

display(Markdown("### Resumen del entrenamiento"))
resumen_entrenamiento = pd.DataFrame([{
    "Experiencias en memoria": len(modelo_dqn.memoria),
    "Actualizaciones de gradiente": modelo_dqn.actualizaciones,
    "Sincronizaciones de la red objetivo": modelo_dqn.sincronizaciones
}])
display(resumen_entrenamiento)

display(Markdown("#### Últimos cinco episodios"))
ultimos = modelo_dqn.registro.tail().copy()
ultimos.columns = ["Episodio", "Epsilon", "Recompensa (pb)", "Experiencias en memoria"]
display(ultimos.reset_index(drop=True).round({"Epsilon": 4, "Recompensa (pb)": 2}))

registro = modelo_dqn.registro

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(registro["episodio"], registro["recompensa_pb"], alpha=.22, color="#2563eb")
axes[0].plot(registro["episodio"], registro["recompensa_pb"].rolling(30, min_periods=1).mean(),
             color="#2563eb", lw=2, label="Media móvil (30 episodios)")
axes[0].set(title="Recompensa de entrenamiento", xlabel="Episodio", ylabel="Recompensa ajustada (pb)")
axes[0].legend(fontsize=8)
axes[1].plot(registro["episodio"], registro["epsilon"], color="#7c3aed", lw=2)
axes[1].set(title="Exploración epsilon", xlabel="Episodio", ylabel="Probabilidad de explorar")
perdidas = pd.Series(modelo_dqn.perdidas)
axes[2].plot(perdidas.index + 1, perdidas, alpha=.2, color="#f59e0b")
axes[2].plot(perdidas.index + 1, perdidas.rolling(100, min_periods=1).mean(),
             color="#d97706", lw=2, label="Media móvil (100)")
axes[2].set(title="Pérdida de Huber", xlabel="Actualización", ylabel="Pérdida")
axes[2].legend(fontsize=8)
for ax in axes:
    ax.grid(alpha=.2)
plt.tight_layout()
plt.show()

auditoria_dqn = auditar(modelo_dqn)

display(Markdown("### Auditoría de una experiencia"))
nombres_estado = [
    "Régimen: DOWN", "Régimen: FLAT", "Régimen: UP",
    "Wallets: BUY", "Wallets: SELL", "Wallets: HOLD", "Wallets: NEUTRAL",
    "Posición (0 = USDC, 1 = POL)", "Etapa normalizada"
]
tabla_estados = pd.DataFrame({
    "Componente": nombres_estado,
    "Estado actual": auditoria_dqn["estado"],
    "Estado siguiente": auditoria_dqn["estado_siguiente"]
})
display(tabla_estados.round(3))
display(Markdown(
    "Las categorías se representan con 1 cuando están activas y 0 en caso contrario. "
    "La posición y la etapa describen la situación del agente."
))

display(Markdown("#### Valores Q de las tres acciones"))
acciones = ["BUY_POL", "SELL_POL", "HOLD"]
tabla_q_auditoria = pd.DataFrame({
    "Acción": acciones,
    "Q actual (pb)": auditoria_dqn["Q_actual"],
    "Q siguiente, red objetivo (pb)": auditoria_dqn["Q_objetivo"],
    "¿Acción ejecutada?": [
        "Sí" if a == auditoria_dqn["accion"] else "No" for a in acciones
    ],
    "¿Válida en el estado siguiente?": [
        "Sí" if a in auditoria_dqn["acciones_futuras_validas"] else "No"
        for a in acciones
    ]
})
display(tabla_q_auditoria.round(2))

q_siguiente_valido = max(
    auditoria_dqn["Q_objetivo"][acciones.index(a)]
    for a in auditoria_dqn["acciones_futuras_validas"]
)
tabla_calculo = pd.DataFrame([
    ("Acción ejecutada", auditoria_dqn["accion"]),
    ("Recompensa inmediata (pb)", f"{auditoria_dqn['recompensa_pb']:.2f}"),
    ("¿Terminó el episodio?", "Sí" if auditoria_dqn["done"] else "No"),
    ("Q de la acción ejecutada (pb)", f"{auditoria_dqn['Q_accion']:.2f}"),
    ("Máximo Q futuro válido (pb)", f"{q_siguiente_valido:.2f}"),
    ("Target de Bellman (pb)", f"{auditoria_dqn['target_pb']:.2f}"),
    ("Pérdida de Huber", f"{auditoria_dqn['huber']:.4f}")
], columns=["Concepto", "Resultado"])

display(Markdown("#### Cálculo de Bellman y pérdida"))
display(tabla_calculo)
display(Markdown(
    "El máximo futuro solo considera acciones válidas para la posición siguiente. "
    "Si el episodio termina, ese valor se ignora y el target es la recompensa inmediata."
))

pos_actual = int(auditoria_dqn["estado"][7])
validas_actuales = {"BUY_POL", "HOLD"} if pos_actual == 0 else {"SELL_POL", "HOLD"}
colores = ["#f59e0b" if a == auditoria_dqn["accion"] else
           "#2563eb" if a in validas_actuales else "#cbd5e1" for a in acciones]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
bars = axes[0].bar(acciones, auditoria_dqn["Q_actual"], color=colores)
axes[0].bar_label(bars, fmt="%.2f", padding=3, fontsize=9)
axes[0].axhline(0, color="#64748b", linewidth=.8)
axes[0].set(title="Valores Q del estado actual", ylabel="Valor Q estimado (pb)")
bars = axes[1].bar(["Q de acción ejecutada", "Target de Bellman"],
                   [auditoria_dqn["Q_accion"], auditoria_dqn["target_pb"]],
                   color=["#2563eb", "#f59e0b"])
axes[1].bar_label(bars, fmt="%.2f", padding=3, fontsize=9)
axes[1].axhline(0, color="#64748b", linewidth=.8)
axes[1].set(title="Predicción frente al target", ylabel="Valor (pb)")
for ax in axes:
    ax.grid(axis="y", alpha=.2)
    ax.margins(y=.2)
plt.tight_layout()
plt.show()

display(Markdown(
    "En la gráfica de valores Q, **naranja** identifica la acción ejecutada; "
    "**azul**, otra acción válida; y **gris**, una acción no admisible. "
    "Durante entrenamiento, la acción ejecutada pudo elegirse por exploración."
))


## 5. Replay histórico común: cuatro políticas en los mismos ejes

Este recorrido compara **DQN, Q-learning, Bellman y Buy and Hold** sobre la misma trayectoria horaria de POL, con capital inicial de 100, posición inicial en USDC y la misma contabilidad económica neta de gas. Los tres modelos se entrenan con esa misma trayectoria y luego se congelan para recorrerla: es una comparación **dentro de muestra**, útil para interpretar decisiones, no una prueba de generalización.

Se incluyen hasta 24 horas, pero solo cuando el final de cada hora está cubierto por los swaps disponibles. Si la última hora del cache está incompleta, se excluye y se indica el número real de horas. Los modelos conservan la posición entre horas; no se reinicia el portafolio cada cuatro horas.

La prueba reservada de cuatro horas se conserva en la sección siguiente. Usa modelos entrenados en bloques de cuatro horas y responde a otra pregunta; no debe compararse directamente con los resultados de este replay.


In [ ]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
from lib_rl.dqn import comparar_replay_historico

replay_comun = comparar_replay_historico(
    observaciones_rl, resultado.swaps_logicos,
    max_horas=24, episodios_dqn=600, episodios_tabla=2000, seed=7
)
trayectorias = replay_comun["resultados"]
n_horas = len(replay_comun["observaciones"])
inicio = replay_comun["observaciones"].as_of.iloc[0]
fin = replay_comun["observaciones"].as_of.iloc[-1] + pd.Timedelta(hours=1)
print(f"Replay dentro de muestra: {n_horas} horas completas | {inicio} → {fin}")
if n_horas < 24:
    print("La última hora incompleta del cache no se usa como si estuviera completa.")

colores = {"DQN":"#f59e0b", "Q-learning":"#2563eb",
           "Bellman":"#7c3aed", "Buy and Hold":"#64748b"}
marcadores = {"Buy and Hold": "D", "Bellman": "^", "DQN": "o", "Q-learning": "s"}
fig, ax = plt.subplots(figsize=(12, 5))
for i, metodo in enumerate(("Buy and Hold", "Bellman", "DQN", "Q-learning")):
    grupo = trayectorias[trayectorias["metodo"] == metodo].sort_values("t")
    ax.plot([0] + (grupo["t"] + 1).tolist(), [100.] + grupo["wealth"].tolist(),
            marker=marcadores[metodo], markevery=(i, 4), markersize=5, linewidth=2,
            color=colores[metodo], label=metodo)
ax.axhline(100, linestyle=":", color="#94a3b8")
ax.set(title=f"Replay histórico común de POL: {n_horas} horas · dentro de muestra",
       xlabel="Horas transcurridas desde el inicio del replay",
       ylabel="Capital económico (inicial = 100)")
ax.legend(title="Política")
ax.grid(alpha=.2)
plt.tight_layout()
plt.show()

print("Cuando dos políticas tienen el mismo capital, sus líneas se superponen. Los marcadores ayudan a identificarlas.")
display(replay_comun["metricas"])


## 5.1 Evaluación con datos reservados

Para evaluar fuera de muestra se conservan cuatro horas que no se usan para entrenar las políticas de este experimento. La tabla muestra el retorno económico neto de gas y las demás métricas. Estos modelos son distintos de los entrenados para el replay histórico completo; cuatro horas no bastan para concluir que los resultados se repitan en el futuro.


In [ ]:
from IPython.display import display, Markdown

resultados_dqn = experimento_dqn["resultados"]
metricas_dqn = resumen_resultados(resultados_dqn)
display(Markdown("**Resultados en las cuatro horas reservadas**, netos de gas. "
                 "Este test usa modelos entrenados por separado del replay histórico."))
display(metricas_dqn)
display(resultados_dqn[resultados_dqn["metodo"] == "DQN"])


## 5.2. ¿Qué hizo el agente? Acciones y transiciones en train y test

Analizamos el **mismo DQN del experimento con test reservado de cuatro horas**. En train mostramos todas las decisiones registradas mientras la red aprendía: el agente explora con epsilon y vuelve a recorrer episodios de cuatro pasos. Por eso son decisiones repetidas de la historia, no horas diferentes. En test la red elegida está congelada y actúa sin exploración; se observan solamente cuatro decisiones.

El estado se escribe como `(régimen de mercado, señal de wallets, posición antes de actuar)`; 0 significa USDC y 1 significa POL. La etapa también es una entrada de la red y se muestra como `t`. El gráfico de transiciones resume **solo el régimen**, mientras las tablas muestran el estado completo. Al terminar cada episodio no se inventa un estado siguiente: la última acción sí cambia o mantiene la posición, pero se marca como terminal.

Las reglas del portafolio restringen las decisiones: con USDC se puede comprar POL o mantener; con POL se puede vender o mantener. En test el agente elige el mayor valor Q entre las acciones permitidas. Q estima recompensa acumulada del objetivo, **no una probabilidad ni una ganancia garantizada**.


In [ ]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from lib_rl.explicacion_dqn import (analizar_decisiones_dqn,
                                    matriz_transiciones_regimen,
                                    graficar_decisiones_dqn)

analisis_acciones = analizar_decisiones_dqn(experimento_dqn)
train_dqn = analisis_acciones["train"]
test_dqn = analisis_acciones["test"]
fig_acciones, fig_transiciones = graficar_decisiones_dqn(analisis_acciones)
display(fig_acciones)
display(fig_transiciones)
plt.close(fig_acciones)
plt.close(fig_transiciones)

display(Markdown("**Cantidad de acciones.** Los porcentajes permiten comparar tamaños de muestra distintos."))
display(analisis_acciones["acciones"])
display(Markdown(f"**Train:** {len(train_dqn)} decisiones en {train_dqn['episodio'].nunique()} episodios; "
                 f"{100 * train_dqn['exploracion'].mean():.1f}% fueron elecciones por exploración. "
                 "Las demás se eligieron con la red disponible en ese momento."))
display(Markdown("**Transiciones del régimen.** Cada celda cuenta pasos del régimen de la fila al de la columna. "
                 "Las escalas de color son independientes y se excluyen los pasos terminales."))
display(matriz_transiciones_regimen(train_dqn).rename_axis("Train: régimen actual"))
display(matriz_transiciones_regimen(test_dqn).rename_axis("Test: régimen actual"))

ultimo = train_dqn[train_dqn["episodio"] == train_dqn["episodio"].max()]
display(Markdown("**Ejemplo de train: último episodio.** `exploracion=True` indica elección aleatoria entre acciones válidas."))
display(ultimo[["episodio", "bloque", "t", "as_of", "estado", "accion", "posicion_despues",
                "estado_siguiente", "done", "epsilon", "exploracion"]])
display(Markdown("**Todas las decisiones de test.** Q de la acción elegida y de la otra acción válida se expresan en puntos básicos del objetivo."))
tabla_test = test_dqn[["t", "as_of", "estado", "acciones_validas", "accion", "posicion_despues",
                       "estado_siguiente", "done", "Q_accion", "Q_alternativa"]].copy()
tabla_test[["Q_accion", "Q_alternativa"]] = tabla_test[["Q_accion", "Q_alternativa"]].round(2)
display(tabla_test)


## 6. Escenario de estrés: señal neutral

Se sustituyen señal y apoyos de wallets por cero en el test. Las políticas siguen congeladas y cada método se evalúa sobre el mismo escenario. Esto mide sensibilidad a la información de wallets, no reproduce una caída real de la fuente de datos.


In [ ]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
import numpy as np

base = resumen_resultados(experimento_dqn["resultados"]).assign(escenario="Base")
estres = resumen_resultados(experimento_dqn["sin_wallets"]).assign(escenario="Sin señal de wallets")
comparacion_escenarios = pd.concat([base, estres], ignore_index=True)

metodos = base["Método"].tolist()
y = np.arange(len(metodos))
valores_base = 100 * base.set_index("Método").loc[metodos, "Retorno económico"].to_numpy()
valores_estres = 100 * estres.set_index("Método").loc[metodos, "Retorno económico"].to_numpy()

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.barh(y + .2, valores_base, height=.36, color="#2563eb", label="Señal de wallets observada")
b2 = ax.barh(y - .2, valores_estres, height=.36, color="#f59e0b", label="Sin señal de wallets")
ax.bar_label(b1, fmt="%+.2f%%", padding=3, fontsize=9)
ax.bar_label(b2, fmt="%+.2f%%", padding=3, fontsize=9)
ax.set_yticks(y, metodos)
ax.axvline(0, color="#64748b", linewidth=.8)
ax.set(title="¿Qué cambia si quitamos la señal de wallets?",
       xlabel="Retorno económico neto en test (%)")
ax.legend(loc="upper center", bbox_to_anchor=(.5, -.20), ncol=2, frameon=False)
ax.grid(axis="x", alpha=.2)
ax.margins(x=.25)
plt.tight_layout()
plt.show()

print("Mismas horas de test y mismas políticas congeladas; no se reentrenó.")
display(comparacion_escenarios)


## 7. Limitaciones de los datos, los modelos y la evaluación

Las limitaciones de esta sección se organizan según su origen. Algunas afectan a toda la comparación y otras corresponden a un modelo específico.

### 7.1 Limitaciones de los datos y del problema

Estas condiciones afectan a Bellman, Q-learning y DQN:

- **Poca historia:** disponemos de una ventana cercana a 24 horas de un solo pool POL/USDC. Repetir sus bloques durante el entrenamiento no crea nuevos datos independientes ni permite representar muchos regímenes de mercado.
- **Señales de wallets:** la función de señales intenta usar perfiles cuyos resultados estén disponibles hasta cada corte. Sin embargo, el ledger incluye reconstrucción FIFO y operaciones HOLD; falta auditar cuándo se conoce cada evento y su resultado, especialmente en límites y cierres.
- **Costos y ejecución:** el costo de gas se aproxima con la mediana de swaps anteriores a validación. No se modelan cambios futuros del gas, slippage, profundidad de mercado ni efectos del tamaño de la orden.
- **Alcance del portafolio:** se usa una posición binaria de inversión completa en USDC o POL, sin ventas en corto ni financiación. Esto simplifica la operación real.
- **Recompensa:** el bonus asociado a las wallets modifica el objetivo del agente, pero no es dinero adicional. Por eso el notebook reporta por separado el retorno económico neto y el objetivo ajustado.

### 7.2 Limitaciones específicas de cada modelo

**Bellman:** estima transiciones y recompensas promedio con muy pocas observaciones. El suavizado de Laplace y los estados no observados pueden influir mucho en la política. En esta comparación se ajusta el MDP con los bloques de entrenamiento; no se reutilizan los valores calculados con toda la ventana en la Parte III.

**Q-learning tabular:** repite los bloques disponibles durante el entrenamiento. Muchos pares estado–acción–etapa pueden recibir pocas visitas o ninguna, de modo que algunos valores Q siguen siendo poco informativos. En esos casos, la acción elegida puede depender del desempate.

**DQN:** comparte parámetros entre estados, lo que permite generalizar, pero con tan pocos datos también puede sobreajustarse, extrapolar mal o producir valores Q imprecisos. Una pérdida baja no garantiza un retorno económico positivo. La elección de epsilon con un solo bloque de validación añade incertidumbre a la configuración seleccionada.

### 7.3 Limitaciones de la evaluación y comparación

- **Horizonte reducido:** para separar entrenamiento, validación y test usamos bloques de cuatro horas. Esta comparación no evalúa directamente los replay completo de las Partes III y IV.
- **Test pequeño:** un único bloque de cuatro horas no permite establecer rentabilidad futura ni diferencias estadísticamente sólidas entre los métodos.
- **Escenario de estrés:** retirar la señal de wallets permite observar sensibilidad en el mismo test, pero no constituye una prueba independiente ni reproduce todas las consecuencias de una falla real de la fuente de datos.
- **Trabajo posterior:** reunir más historia, fijar los períodos y configuraciones antes del test y repetir la evaluación en distintas ventanas, regímenes y costos. Con datos suficientes se podrían comparar episodios independientes de 24 horas mediante validación temporal walk-forward.

- **Orden de transacciones:** el cache no conserva la posición exacta de la transacción dentro del bloque. Para empates de tiempo se usa un orden reproducible por bloque y hash, que no garantiza el orden real del FIFO.
